# Two-Scales Estimation

The multivariate two-scales estimator of [Zhang (2011)](https://doi.org/10.1016/j.jeconom.2010.03.012) extends the two-scales volatility of [Zhang, Mykland and Aït-Sahalia (2005)](https://doi.org/10.1198/016214505000000169) to covariances: it combines realized covariance on a slow time scale, averaged over offset sub-grids, with realized covariance on every tick, which measures the noise. This tutorial runs it with `estimate_covariance(ts, assets, :two_scales_covariance)` and varies the number of sub-grids.

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## The Simulated Assets

Four assets with asynchronous ticks, noise and a known true covariance, as in the first tutorial.

In [ ]:
using HighFrequencyCovariance, LinearAlgebra, Random, Statistics
import Plots

ts, true_covar, true_noise, _ = generate_random_path(4, 10000; rng = MersenneTwister(2))  # four assets, 10,000 ticks in all, known true covariance
assets = get_assets(ts)

## The Two-Scales Estimate

With no `num_grids`, the package uses `default_num_grids(ts)`, the fewest ticks of any asset divided by 100 and at least 3.

In [ ]:
tsrc = estimate_covariance(ts, assets, :two_scales_covariance)
(K = default_num_grids(ts), two_scales = calculate_mean_abs_distance(tsrc, true_covar), realized = calculate_mean_abs_distance(estimate_covariance(ts, assets, :simple_covariance), true_covar))

## The Number of Sub-grids

More sub-grids lengthen the slow time scale, which lowers the noise left in it and lets the less traded assets trade within each sub-grid return.

In [ ]:
Ks = [3, 5, 11, 20, 40, 80, 150]
err = [calculate_mean_abs_distance(estimate_covariance(ts, assets, :two_scales_covariance; num_grids = k), true_covar) for k in Ks]
Plots.plot(Ks, [e.Correlation_error for e in err]; marker = :circle, xlabel = "sub-grids K", ylabel = "correlation error", label = "two-scales", title = "Two-scales covariance against K")

## The Noise of Each Asset

The fast scale also gives each asset's noise variance, with `estimate_microstructure_noise`, here set against the variance the simulation used.

In [ ]:
est_noise = estimate_microstructure_noise(ts, assets)
[(asset = a, estimated = est_noise[a], true_value = true_noise[a]) for a in assets]

## Weights of the Combination

The estimator combines two linear combinations of each pair of assets. `equalweight = true` weights them equally instead of by the volatilities.

In [ ]:
(optimal_weights = calculate_mean_abs_distance(tsrc, true_covar), equal_weights = calculate_mean_abs_distance(estimate_covariance(ts, assets, :two_scales_covariance; equalweight = true), true_covar))

## Things to Try

- Draw a path with more ticks and see how the best K grows.
- Check whether the raw two-scales estimate, with `regularisation = missing`, is positive semi-definite.